# LouisFarm — Semaine 8 : Series Temporelles & Prevision
## Dataset : Prix mensuel du cacao — Cote d Ivoire (2010-2024)

**Objectif :** Analyser des donnees temporelles et construire des modeles de prevision.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns; sns.set_theme(style="whitegrid")
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings; warnings.filterwarnings("ignore")
import sys; sys.path.insert(0, ".")
from utils_louisfarm import gen_cacao_ci_timeseries

df = gen_cacao_ci_timeseries()
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()
ts = df["prix_tonne_usd"]

print(f"Serie temporelle: {len(ts)} mois ({ts.index[0].strftime('%Y-%m')} a {ts.index[-1].strftime('%Y-%m')})")
print(f"Prix USD - Min:{ts.min():.0f} | Max:{ts.max():.0f} | Moy:{ts.mean():.0f}")

## Lecon 8.1 — Decomposition d une serie temporelle

In [ ]:
# DECOMPOSITION DE LA SERIE
print("DECOMPOSITION DE LA SERIE TEMPORELLE")
print("Cacao Cote d Ivoire — Prix mensuel USD/tonne")
print("=" * 52)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
fig.suptitle("Semaine 8 — Decomposition : Prix cacao CI 2010-2024", fontweight="bold")

result = seasonal_decompose(ts, model="additive", period=12)
labels = ["Serie originale", "Tendance", "Saisonnalite", "Residu"]
components = [ts, result.trend, result.seasonal, result.resid]
colors = ["#2E86AB", "#C73E1D", "#F18F01", "#888888"]

for ax, comp, label, col in zip(axes, components, labels, colors):
    ax.plot(comp.index, comp.values, color=col, linewidth=1.5)
    ax.set_ylabel(label, fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    if label == "Tendance":
        ax.set_ylabel("Tendance\n(hausse long terme)", fontsize=9)
    if label == "Saisonnalite":
        ax.axhline(0, color="black", linestyle="--", alpha=0.3)

axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.savefig("./s8_decomposition.png", dpi=100, bbox_inches="tight")
plt.show()

trend_slope = (result.trend.dropna().iloc[-1] - result.trend.dropna().iloc[0]) / len(result.trend.dropna())
print(f"Tendance mensuelle estimee: +{trend_slope:.1f} USD/mois")

## Lecon 8.2 — Stationnarite et ACF/PACF

In [ ]:
# TEST DE STATIONNARITE (ADF)
print("TEST DE STATIONNARITE — Augmented Dickey-Fuller")
print("=" * 55)

def test_adf(series, name=""):
    result = adfuller(series.dropna(), autolag="AIC")
    print(f"  {name}:")
    print(f"    Statistique ADF : {result[0]:.4f}")
    print(f"    P-value         : {result[1]:.4f}")
    print(f"    Valeurs critiques: {result[4]}")
    stationnaire = "STATIONNAIRE (p<0.05)" if result[1] < 0.05 else "NON STATIONNAIRE (p>=0.05)"
    print(f"    => {stationnaire}")
    return result[1] < 0.05

print("1. Serie originale :")
is_stat = test_adf(ts, "Prix USD/tonne")

if not is_stat:
    print("\n2. Serie differenciee (ordre 1) :")
    ts_diff = ts.diff().dropna()
    is_stat_diff = test_adf(ts_diff, "Prix diff(1)")
    
    if not is_stat_diff:
        print("\n3. Serie double-differenciee :")
        ts_diff2 = ts_diff.diff().dropna()
        test_adf(ts_diff2, "Prix diff(2)")

# ACF et PACF
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Semaine 8 — ACF et PACF pour identifier p,q", fontweight="bold")
plot_acf(ts.diff().dropna(), lags=24, ax=axes[0], title="ACF (serie differenciee)", alpha=0.05)
plot_pacf(ts.diff().dropna(), lags=24, ax=axes[1], title="PACF (serie differenciee)", alpha=0.05, method="ywm")
axes[0].set_xlabel("Lag (mois)"); axes[1].set_xlabel("Lag (mois)")
plt.tight_layout()
plt.savefig("./s8_acf_pacf.png", dpi=100, bbox_inches="tight")
plt.show()
print("\nInterpretation ACF/PACF: choisir p (ordre AR) et q (ordre MA)")

## Lecon 8.3 — Modeles AR, ARIMA : construction et comparaison

In [ ]:
# SPLIT TRAIN/TEST (temporel: jamais melanger future/passe!)
n_test = 18  # 18 derniers mois pour le test
ts_train = ts.iloc[:-n_test]
ts_test  = ts.iloc[-n_test:]
print(f"Train: {len(ts_train)} mois | Test: {len(ts_test)} mois")

resultats = {}

# MODELE 1: AutoRegression (AR)
model_ar = AutoReg(ts_train, lags=3)
fit_ar = model_ar.fit()
pred_ar = fit_ar.forecast(steps=n_test)
mae_ar = mean_absolute_error(ts_test, pred_ar)
rmse_ar = np.sqrt(mean_squared_error(ts_test, pred_ar))
resultats["AR(3)"] = {"MAE": mae_ar, "RMSE": rmse_ar, "pred": pred_ar}
print(f"AR(3) : MAE={mae_ar:.1f} USD | RMSE={rmse_ar:.1f} USD")

# MODELE 2: ARIMA(1,1,1)
model_arima111 = ARIMA(ts_train, order=(1,1,1))
fit_111 = model_arima111.fit()
pred_111 = fit_111.forecast(steps=n_test)
mae_111 = mean_absolute_error(ts_test, pred_111)
rmse_111 = np.sqrt(mean_squared_error(ts_test, pred_111))
resultats["ARIMA(1,1,1)"] = {"MAE": mae_111, "RMSE": rmse_111, "pred": pred_111}
print(f"ARIMA(1,1,1): MAE={mae_111:.1f} USD | RMSE={rmse_111:.1f} USD")

# MODELE 3: ARIMA(2,1,2)
model_arima212 = ARIMA(ts_train, order=(2,1,2))
fit_212 = model_arima212.fit()
pred_212 = fit_212.forecast(steps=n_test)
mae_212 = mean_absolute_error(ts_test, pred_212)
rmse_212 = np.sqrt(mean_squared_error(ts_test, pred_212))
resultats["ARIMA(2,1,2)"] = {"MAE": mae_212, "RMSE": rmse_212, "pred": pred_212}
print(f"ARIMA(2,1,2): MAE={mae_212:.1f} USD | RMSE={rmse_212:.1f} USD")

# BASELINE: Naive (derniere valeur connue)
naive_pred = pd.Series([ts_train.iloc[-1]] * n_test, index=ts_test.index)
mae_naive = mean_absolute_error(ts_test, naive_pred)
print(f"\nBaseline naive: MAE={mae_naive:.1f} USD")
print(f"Meilleur modele: {min(resultats, key=lambda k: resultats[k]['MAE'])}")

## Lecon 8.4 — Previsions avec intervalles de confiance

In [ ]:
# VISUALISATION ET PREVISIONS
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle("Semaine 8 — Modeles de prevision : Prix cacao CI", fontweight="bold")

# Plot 1: Comparaison sur la periode de test
axes[0].plot(ts_train.index[-36:], ts_train.iloc[-36:], color="#2E86AB", linewidth=2, label="Observe (train)")
axes[0].plot(ts_test.index, ts_test.values, color="#2E86AB", linewidth=2, linestyle="--", label="Observe (test)")
colors_m = {"AR(3)":"#C73E1D","ARIMA(1,1,1)":"#F18F01","ARIMA(2,1,2)":"#3B1F2B"}
for name, res in resultats.items():
    axes[0].plot(ts_test.index, res["pred"], linewidth=1.5, label=f"{name} (MAE={res['MAE']:.0f}$)", color=colors_m[name])
axes[0].set_title("Predictions vs Observations (18 derniers mois)")
axes[0].set_ylabel("Prix USD/tonne"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Plot 2: Previsions 6 mois avec intervalles de confiance
best_model_name = min(resultats, key=lambda k: resultats[k]["MAE"])
order = {"AR(3)":(3,0,0),"ARIMA(1,1,1)":(1,1,1),"ARIMA(2,1,2)":(2,1,2)}.get(best_model_name, (1,1,1))
best_fit = ARIMA(ts, order=order).fit()

forecast_result = best_fit.get_forecast(steps=6)
forecast_mean = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int(alpha=0.05)

axes[1].plot(ts.index[-24:], ts.iloc[-24:], color="#2E86AB", linewidth=2, label="Historique")
axes[1].plot(forecast_mean.index, forecast_mean.values, color="#C73E1D", linewidth=2.5, label="Prevision")
axes[1].fill_between(forecast_ci.index, forecast_ci.iloc[:,0], forecast_ci.iloc[:,1],
                     alpha=0.25, color="#C73E1D", label="IC 95%")
axes[1].set_title(f"Prevision 6 mois ({best_model_name}) — Prix cacao CI")
axes[1].set_ylabel("Prix USD/tonne"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("./s8_forecast.png", dpi=100, bbox_inches="tight")
plt.show()

print("PREVISIONS POUR LES 6 PROCHAINS MOIS :")
print("=" * 50)
for date, val, ci_low, ci_high in zip(forecast_mean.index, forecast_mean.values,
                                       forecast_ci.iloc[:,0], forecast_ci.iloc[:,1]):
    print(f"  {date.strftime('%Y-%m')} : {val:>7.0f} USD/t  (IC: [{ci_low:.0f} - {ci_high:.0f}])")
    
print()
print("NOTE AUX COOPERATIVES DE LA BCC CI:")
print(f"  Le modele {best_model_name} predit une stabilisation des prix")
print(f"  dans une fourchette de {forecast_ci.iloc[:,0].mean():.0f} a {forecast_ci.iloc[:,1].mean():.0f} USD/tonne.")
print(f"  L incertitude augmente au-dela de 3 mois.")
print(f"  Nous recommandons de securiser des contrats a terme a <3 mois.")

## API pour donnees de marche (a utiliser localement)

```python
# Yahoo Finance (pip install yfinance)
import yfinance as yf
cacao = yf.download('CC=F', start='2010-01-01', end='2024-06-01', interval='1mo')
print(cacao.head())

# Alpha Vantage (cle gratuite sur alphavantage.co)
import requests
url = 'https://www.alphavantage.co/query'
params = {'function':'TIME_SERIES_MONTHLY','symbol':'COPX','apikey':'VOTRE_CLE'}
r = requests.get(url, params=params)
```